# How does the spike detector work?　/　スパイク検出のしくみ

> **The one question behind everything / すべての出発点となる問い**
>
> *"How surprised should we be by this minute's traffic?"*
> *「この1分間のアクセス数は、どのくらい"ありえない"数字なのか?」*

**EN** — Imagine a call center that normally gets about 8 calls a minute. If 40 calls arrive in one minute you don't need statistics to know something happened. But what about 14? Or 18? This notebook shows, step by step and with sliders, how the analyzer puts a number on "unusual". No statistics background needed.

**JA** — 普段は1分あたり約8件の電話がかかってくるコールセンターを想像してください。1分間に40件かかってくれば、統計学がなくても「何かあった」とわかります。では14件は? 18件は? このノートブックでは、スライダーを動かしながら、分析ツールが「いつもと違う」を数字にする手順を順番に見ていきます。統計の知識は不要です。

| Step | English | 日本語 |
|---|---|---|
| 1 | Normal traffic is noisy | 普通のアクセスもゆらぐ |
| 2 | The average is the "normal level" (λ) | 平均が「ふだんの水準」(λ) |
| 3 | Random arrivals follow a predictable shape | ランダムな到着には予測できる形がある |
| 4 | A p-value = "how rare is this?" | p値 =「どのくらい珍しい?」 |
| 5 | Checking many minutes causes false alarms | たくさんの分を調べると誤報が出る |
| 6 | A huge burst hides smaller ones, so re-fit the baseline | 巨大な山は小さな山を隠すので、基準を引き直す |
| 7 | Dispersion: can we trust the alerts? | 散らばり指数: 警告を信用してよい? |
| 8 | Play with the real pipeline | 実際のパイプラインで遊ぶ |

*Plot labels are English only (font availability); all explanations are bilingual. / グラフの文字は(フォントの都合で)英語のみです。説明文は日英併記です。*

## Setup　/　準備

Needs `pip install -e ".[notebook]"`. / `pip install -e ".[notebook]"` が必要です。

In [1]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import Dropdown, IntSlider, FloatSlider, interact
from scipy.stats import poisson

# Make the package importable from the repository root or from notebooks/.
repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
sys.path.insert(0, str(repo_root / "src"))

from poisson_log_analyzer import bin_counts, detect_spikes

plt.rcParams.update({"figure.figsize": (10, 3.5), "axes.grid": True, "grid.alpha": 0.25})

## 1. Normal traffic is noisy　/　普通のアクセスもゆらぐ

**EN** — Below, *nothing unusual happens*: every minute has the same average traffic. Yet the numbers jump around. So a single high number is not, by itself, a spike. Move **resample** to get a different "random day".

**JA** — 下のグラフでは*何も異常は起きていません*。どの分も平均は同じです。それでも数字はバラバラに上下します。つまり、1つ大きな数字があるだけでは「スパイク」とは言えません。**resample** を動かすと、別の「ランダムな1日」が見られます。

In [2]:
@interact(
    rate=IntSlider(8, 2, 50, description="average", continuous_update=False),
    seed=IntSlider(1, 0, 99, description="resample", continuous_update=False),
)
def show_noise(rate=8, seed=1):
    counts = np.random.default_rng(seed).poisson(rate, 120)
    fig, ax = plt.subplots()
    ax.plot(counts, marker="o", markersize=3, color="steelblue")
    ax.set(
        title=f"Requests per minute, average {rate}: nothing unusual happened",
        xlabel="Minute", ylabel="Requests",
    )
    plt.show()
    print(f"EN: quietest minute = {counts.min()}, busiest minute = {counts.max()}, with the same average.")
    print(f"JA: 平均は同じでも、最小 {counts.min()} 件〜最大 {counts.max()} 件までばらつきました。")

interactive(children=(IntSlider(value=8, continuous_update=False, description='average', max=50, min=2), IntSl…

## 2. The average is the "normal level" (λ)　/　平均が「ふだんの水準」(λ)

**EN** — The dashed line is the **average**, called **λ** (lambda): the "normal level". Random traffic wobbles around it by roughly **√λ**; the shaded band is *average ± 3 × √λ*. Drag **extra requests** to add a burst at minute 60. It only looks alarming once it clearly leaves the band.

**JA** — 破線は**平均**で、**λ**(ラムダ)と呼びます。これが「ふだんの水準」です。ランダムなアクセスはその周りをおよそ **√λ** ほどゆらぎます。帯は *平均 ± 3 × √λ* の範囲です。**extra requests** を動かして60分目に山を足してみましょう。帯をはっきり超えて初めて「おかしい」と感じられます。

In [3]:
@interact(
    extra=IntSlider(12, 0, 40, description="extra requests", continuous_update=False),
    rate=IntSlider(8, 2, 50, description="average", continuous_update=False),
)
def show_level(extra=12, rate=8):
    counts = np.random.default_rng(1).poisson(rate, 120)
    counts[60] += extra
    wobble = 3 * np.sqrt(rate)
    fig, ax = plt.subplots()
    ax.axhspan(rate - wobble, rate + wobble, color="gray", alpha=0.15, label="typical wobble: average ± 3√λ")
    ax.axhline(rate, color="gray", linestyle="--", label="average (λ)")
    ax.plot(counts, marker="o", markersize=3, color="steelblue", label="requests per minute")
    ax.annotate("minute 60", (60, counts[60]), textcoords="offset points", xytext=(8, 4))
    ax.set(xlabel="Minute", ylabel="Requests")
    ax.legend(loc="upper left")
    plt.show()

interactive(children=(IntSlider(value=12, continuous_update=False, description='extra requests', max=40), IntS…

## 3. Random arrivals follow a predictable shape　/　ランダムな到着には予測できる形がある

**EN** — When requests arrive independently at random, the number per minute follows the **Poisson distribution**, whose shape is fixed by the average λ alone. Blue bars are simulated minutes; the red curve is the prediction. Add more minutes and the bars hug the curve. This is the "model" in *poisson*-log-analyzer.

**JA** — リクエストが互いに無関係にランダムに届くとき、1分あたりの件数は**ポアソン分布**に従い、その形は平均 λ だけで決まります。青い棒はシミュレーションした各分の件数、赤い線は理論上の予測です。分数を増やすと棒が曲線にぴったり重なっていきます。これが *poisson*-log-analyzer の「モデル」です。

In [4]:
@interact(
    rate=IntSlider(8, 2, 50, description="average", continuous_update=False),
    minutes=IntSlider(500, 20, 5000, 20, description="minutes", continuous_update=False),
)
def show_shape(rate=8, minutes=500):
    counts = np.random.default_rng(1).poisson(rate, minutes)
    top = max(int(counts.max()), rate + int(5 * np.sqrt(rate)))
    ks = np.arange(0, top + 1)
    fig, ax = plt.subplots()
    ax.hist(counts, bins=np.arange(-0.5, top + 1.5), density=True, alpha=0.6, label="simulated minutes")
    ax.plot(ks, poisson.pmf(ks, rate), "o-", color="crimson", markersize=3, label="Poisson prediction")
    ax.set(xlabel="Requests in one minute", ylabel="Share of minutes")
    ax.legend()
    plt.show()

interactive(children=(IntSlider(value=8, continuous_update=False, description='average', max=50, min=2), IntSl…

## 4. A p-value = "how rare is this?"　/　p値 =「どのくらい珍しい?」

**EN** — The **p-value** answers: *"If nothing unusual were going on, how often would a minute be this busy **or busier**?"* It is the total height of the red bars. A small p-value means a rare event, which makes us suspicious. Slide **count k** to the right and watch the red area, and the "once in N minutes" statement, change.

**JA** — **p値**は次の問いへの答えです。*「何も異常がないとき、この件数**以上**になるのは、どのくらいの頻度か?」* 赤い棒の高さの合計がp値です。p値が小さいほど珍しい出来事で、「怪しい」と判断します。**count k** を右へ動かして、赤い面積と「N分に1回」の表示がどう変わるか見てください。

In [5]:
@interact(
    rate=IntSlider(8, 2, 40, description="average", continuous_update=False),
    k=IntSlider(15, 1, 60, description="count k", continuous_update=False),
)
def show_tail(rate=8, k=15):
    top = max(k, rate + int(6 * np.sqrt(rate))) + 5
    ks = np.arange(0, top + 1)
    p = poisson.sf(k - 1, rate)
    fig, ax = plt.subplots()
    ax.bar(ks, poisson.pmf(ks, rate), color=np.where(ks >= k, "crimson", "lightsteelblue"))
    ax.set(
        title=f"p-value = P(count ≥ {k}) = {p:.4g}",
        xlabel="Requests in one minute", ylabel="Probability",
    )
    plt.show()
    once_in = 1 / p if p > 0 else float("inf")
    shown = f"{once_in:,.0f}" if once_in < 1e9 else f"{once_in:.1e}"
    print(f"EN: normal traffic reaches {k}+ requests about once in {shown} minutes.")
    print(f"JA: 通常のアクセスで {k} 件以上になるのは、約 {shown} 分に1回です。")

interactive(children=(IntSlider(value=8, continuous_update=False, description='average', max=40, min=2), IntSl…

## 5. Checking many minutes causes false alarms　/　たくさんの分を調べると誤報が出る

**EN** — Suppose we flag every minute with p-value ≤ 0.01 ("1 in 100"). A day has 1,440 minutes, so even on a perfectly normal day about 1,440 × 0.01 ≈ **14 minutes** get flagged by pure chance. That would bury you in false alarms. The analyzer therefore **adjusts for checking many minutes** (the Benjamini–Hochberg correction), so that false alarms stay rare. Below, every flag is false, because the traffic is pure noise. Each bar averages 100 simulated days.

**JA** — p値が0.01以下(「100回に1回」)の分をすべて警告にするとします。1日は1,440分あるので、まったく普通の日でも、偶然だけで約 1,440 × 0.01 ≒ **14分**が警告されてしまいます。これでは誤報だらけです。そこで本ツールは**「たくさんの分を調べたこと」を補正**(Benjamini–Hochberg 法)し、誤報が少なくなるようにしています。下の図ではトラフィックが純粋なノイズなので、警告はすべて誤報です。棒は100日分のシミュレーションの平均です。

In [6]:
@interact(
    minutes=IntSlider(1440, 200, 5000, 100, description="minutes", continuous_update=False),
    alpha=Dropdown(options=[0.001, 0.01, 0.05, 0.1], value=0.01, description="threshold"),
)
def false_alarm_demo(minutes=1440, alpha=0.01):
    trials = 100
    rng = np.random.default_rng(0)
    naive, adjusted = [], []
    for _ in range(trials):
        counts = rng.poisson(8, minutes)
        p = poisson.sf(counts - 1, counts.mean())
        naive.append(int((p <= alpha).sum()))
        adjusted.append(len(detect_spikes(counts, alpha=alpha).flagged))
    means = [np.mean(naive), np.mean(adjusted)]
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.bar(["flag every p ≤ threshold", "analyzer (adjusted)"], means, color=["lightcoral", "seagreen"])
    ax.set(title=f"False alarms per {minutes} normal minutes (average of {trials} runs)", ylabel="False alarms")
    plt.show()
    print(f"EN: naive flagging raised {means[0]:.1f} false alarms per run; the analyzer raised {means[1]:.2f}.")
    print(f"JA: 単純な判定では1回あたり誤報 {means[0]:.1f} 件、本ツールでは {means[1]:.2f} 件でした。")

interactive(children=(IntSlider(value=1440, continuous_update=False, description='minutes', max=5000, min=200,…

## 6. A huge burst hides smaller ones, so re-fit the baseline　/　巨大な山は小さな山を隠すので、基準を引き直す

**EN** — To judge a minute we need λ, the normal level. But if we average *all* minutes, a huge burst drags the average up and makes a smaller burst look normal. The analyzer therefore **flags the obvious minutes, leaves them out, and recomputes λ**, repeating until stable. Here a small burst (minutes 70–79) follows a huge one (minutes 30–34). Raise **huge burst size** and watch the orange line (plain average of all minutes) climb and miss the small burst, while the green line (analyzer baseline) stays near normal and still catches it. At 0, both methods catch the small burst.

**JA** — ある分を判定するには「ふだんの水準」λが必要です。しかし*すべて*の分を平均すると、巨大な山が平均を押し上げ、小さな山を「普通」に見せてしまいます。そこで本ツールは**はっきり怪しい分を除外してλを計算し直す**作業を、値が安定するまで繰り返します。ここでは巨大な山(30〜34分)の後に小さな山(70〜79分)があります。**huge burst size** を上げると、オレンジ(全分の単純な平均)は上がって小さな山を見逃しますが、緑(本ツールの基準)は通常の水準に近いまま、小さな山も検出できます。0のときは、どちらの方法でも小さな山を検出できます。

In [7]:
@interact(
    huge=IntSlider(200, 0, 400, 10, description="huge burst size", continuous_update=False),
)
def show_baseline(huge=200):
    alpha = 0.01
    counts = np.random.default_rng(3).poisson(8, 120)
    counts[30:35] += huge
    counts[70:80] += 16
    small = set(range(70, 80))

    naive_rate = counts.mean()
    naive_flags = set(np.flatnonzero(poisson.sf(counts - 1, naive_rate) <= alpha))
    result = detect_spikes(counts, alpha=alpha)
    robust_flags = set(result.flagged)

    fig, ax = plt.subplots()
    ax.plot(counts, marker="o", markersize=3, color="steelblue", label="requests per minute")
    ax.axhline(naive_rate, color="darkorange", linestyle="--", label=f"plain average of all minutes = {naive_rate:.1f}")
    ax.axhline(result.rate, color="seagreen", linestyle="--", label=f"analyzer baseline = {result.rate:.1f}")
    flagged = sorted(robust_flags)
    ax.scatter(flagged, counts[flagged], color="crimson", zorder=3, label="flagged by analyzer")
    ax.set(xlabel="Minute", ylabel="Requests")
    ax.set_ylim(0, max(60, 1.1 * counts[70:80].max()))
    ax.legend(loc="upper right", fontsize=8)
    plt.show()
    print(f"EN: small-burst minutes caught: plain average {len(small & naive_flags)}/10, analyzer {len(small & robust_flags)}/10.")
    print(f"JA: 小さな山の10分のうち検出できた数: 単純な平均 {len(small & naive_flags)}/10、本ツール {len(small & robust_flags)}/10。")

interactive(children=(IntSlider(value=200, continuous_update=False, description='huge burst size', max=400, st…

## 7. Dispersion: can we trust the alerts?　/　散らばり指数: 警告を信用してよい?

**EN** — The Poisson model assumes requests arrive independently. Real traffic is often *burstier* (bots, retries, campaigns). The **dispersion index** (variance ÷ mean of the normal minutes) checks this: **≈ 1 means the model fits**, and **well above 1 means traffic is wilder than the model expects**, so p-values are too optimistic and false alarms appear. Below, the traffic is always *normal* (no real spike) with the same average. Raise **burstiness** and watch the shape widen and false alarms appear.

**JA** — ポアソンモデルは「リクエストは互いに無関係に届く」と仮定します。実際のアクセスは(ボット、再試行、キャンペーンなどで)もっと*かたまって*届くことがよくあります。**散らばり指数**(通常の分の分散 ÷ 平均)はこれを確かめる指標です。**1付近ならモデルが合っている**ことを、**1よりかなり大きいと、モデルの想定より激しくゆらいでいる**ことを意味し、p値は甘すぎて誤報が出ます。下のトラフィックは平均が同じで、本物のスパイクは*ありません*。**burstiness** を上げると、形が広がり誤報が出てくるのを確認してください。

In [8]:
@interact(
    burstiness=FloatSlider(value=1.0, min=1.0, max=5.0, step=0.25, description="burstiness", continuous_update=False),
    minutes=IntSlider(1440, 200, 5000, 100, description="minutes", continuous_update=False),
)
def show_dispersion(burstiness=1.0, minutes=1440):
    mean = 8
    rng = np.random.default_rng(0)
    if burstiness <= 1.0:
        counts = rng.poisson(mean, minutes)
    else:
        # Negative binomial with the same mean but variance = burstiness * mean.
        counts = rng.negative_binomial(mean / (burstiness - 1), 1 / burstiness, minutes)
    result = detect_spikes(counts, alpha=0.01)
    top = max(int(counts.max()), mean + 25)
    ks = np.arange(0, top + 1)
    fig, ax = plt.subplots()
    ax.hist(counts, bins=np.arange(-0.5, top + 1.5), density=True, alpha=0.6, label="simulated normal minutes")
    ax.plot(ks, poisson.pmf(ks, mean), "o-", color="crimson", markersize=3, label="what the Poisson model expects")
    ax.set(
        title=f"dispersion index = {result.dispersion:.2f}; false alarms = {len(result.flagged)}",
        xlabel="Requests in one minute", ylabel="Share of minutes",
    )
    ax.legend()
    plt.show()
    verdict_en = "model fits, alerts are trustworthy" if result.dispersion < 1.25 else "traffic is burstier than the model assumes, so be careful with alerts"
    verdict_ja = "モデルに合っており、警告は信頼できます" if result.dispersion < 1.25 else "モデルの想定より激しくゆらいでいるため、警告は慎重に扱ってください"
    print(f"EN: {verdict_en}.")
    print(f"JA: {verdict_ja}。")

interactive(children=(FloatSlider(value=1.0, continuous_update=False, description='burstiness', max=5.0, min=1…

## 8. Play with the real pipeline　/　実際のパイプラインで遊ぶ

**EN** — This uses the actual functions: events with timestamps → `bin_counts` (count per window) → `detect_spikes`. A real burst of extra requests starts at minute 50. Try:
- a **short window** is sensitive to short bursts but each count is small and noisy;
- a **long window** averages noise away but can dilute a short burst;
- a **stricter threshold** (smaller α) gives fewer false alarms but may miss weak bursts.

**JA** — ここでは実際の関数を使います。タイムスタンプ付きのイベント → `bin_counts`(窓ごとの件数) → `detect_spikes`。50分目から、追加リクエストによる本物の山が始まります。次を試してください。
- **短い窓**は短い山に敏感ですが、1窓あたりの件数が少なくゆらぎが大きくなります。
- **長い窓**はノイズを平均化しますが、短い山を薄めてしまうことがあります。
- **厳しいしきい値**(小さい α)は誤報が減りますが、弱い山を見逃すことがあります。

In [9]:
@interact(
    window_seconds=Dropdown(options=[10, 30, 60, 120, 300, 600], value=60, description="window (s)"),
    alpha=Dropdown(options=[0.001, 0.01, 0.05, 0.1], value=0.01, description="threshold α"),
    burst_events=IntSlider(150, 0, 600, 10, description="burst size", continuous_update=False),
    burst_seconds=IntSlider(120, 10, 900, 10, description="burst secs", continuous_update=False),
)
def explore(window_seconds=60, alpha=0.01, burst_events=150, burst_seconds=120):
    rng = np.random.default_rng(0)
    t0 = datetime(2025, 1, 1, tzinfo=timezone.utc)
    duration, base_rate, burst_start = 7200, 0.1, 3000  # seconds, events/second, seconds
    normal = rng.uniform(0, duration, rng.poisson(base_rate * duration))
    burst = rng.uniform(burst_start, burst_start + burst_seconds, burst_events)
    stamps = [t0 + timedelta(seconds=float(s)) for s in np.concatenate([normal, burst])]

    start, counts = bin_counts(stamps, window_seconds)
    result = detect_spikes(counts, alpha=alpha)
    offset = (start - t0).total_seconds()
    minutes = (offset + np.arange(len(counts)) * window_seconds) / 60

    fig, ax = plt.subplots()
    ax.axvspan(burst_start / 60, (burst_start + burst_seconds) / 60, color="gold", alpha=0.3, label="real burst")
    ax.step(minutes, counts, where="post", color="steelblue", label=f"requests per {window_seconds}s window")
    ax.axhline(result.rate, color="gray", linestyle="--", label=f"baseline λ = {result.rate:.1f}")
    flagged = np.asarray(result.flagged, dtype=int)
    ax.scatter(minutes[flagged], counts[flagged], color="crimson", zorder=3, label="flagged")
    ax.set(xlabel="Minutes from start", ylabel="Requests per window")
    ax.legend(loc="upper left", fontsize=8)
    plt.show()

    hits = [i for i in result.flagged if burst_start <= offset + i * window_seconds + window_seconds / 2 <= burst_start + burst_seconds]
    false_alarms = len(result.flagged) - len(hits)
    print(f"EN: {len(hits)} flagged window(s) inside the burst, {false_alarms} false alarm(s). Dispersion index: {result.dispersion:.2f}.")
    print(f"JA: 山の中で警告された窓 {len(hits)} 個、誤報 {false_alarms} 個。散らばり指数: {result.dispersion:.2f}。")

interactive(children=(Dropdown(description='window (s)', index=2, options=(10, 30, 60, 120, 300, 600), value=6…

## Recap　/　まとめ

| Idea | Plain English | 日本語 | In the code |
|---|---|---|---|
| λ (lambda) | The normal level: average requests per window | ふだんの水準(1窓あたりの平均件数) | `result.rate` |
| Poisson model | Random independent arrivals have a predictable shape | ランダムで独立した到着がつくる、予測可能な形 | `scipy.stats.poisson` |
| p-value | How often normal traffic is this busy or busier | 通常のアクセスがこれ以上に多くなる頻度 | `result.p_values` |
| Adjusted p-value (q) | p-value corrected for checking many windows | たくさんの窓を調べたことを補正したp値 | `result.q_values` |
| Threshold α | How strict we are about false alarms | 誤報に対する厳しさ | `alpha` |
| Robust baseline | Re-fit λ without the suspicious windows | 怪しい窓を除いてλを引き直す | `detect_spikes` loop |
| Dispersion index | ≈ 1: model fits; ≫ 1: traffic is burstier than assumed | ≈1: モデルに合う、≫1: 想定より激しくゆらぐ | `result.dispersion` |

<details>
<summary>For the curious: the formulas　/　興味のある方へ: 数式</summary>

- Poisson probability of count *k* with average λ: P(X = k) = e<sup>−λ</sup> λ<sup>k</sup> / k!
- p-value of a window with count *k*: P(X ≥ k) = 1 − P(X ≤ k − 1)
- Benjamini–Hochberg: sort the *n* p-values, scale the *i*-th smallest by *n / i*, and keep the running minimum from the largest down. A window is flagged when its q-value ≤ α.
- Dispersion index: variance ÷ mean of the baseline windows (exactly 1 for a Poisson distribution).

- ポアソン分布で、平均λのときに件数がちょうど *k* になる確率: P(X = k) = e<sup>−λ</sup> λ<sup>k</sup> / k!
- 件数 *k* の窓のp値: P(X ≥ k) = 1 − P(X ≤ k − 1)
- Benjamini–Hochberg 法: *n* 個のp値を小さい順に並べ、*i* 番目を *n / i* 倍し、最大側から累積最小値をとります。q値 ≤ α の窓を警告します。
- 散らばり指数: 基準となる窓の分散 ÷ 平均(ポアソン分布ならちょうど1)。

</details>